In [9]:
import os
import pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

base = './'                                    # trained models are saved in ./Models
os.makedirs(base + 'Models', exist_ok=True)

# find the data: next to this notebook, or in the project's "1 - Data Set and processing" folder
for data_dir in ['./', '../1 - Data Set and processing/']:
    if os.path.exists(data_dir + 'phishing_train.csv'):
        break
else:
    raise FileNotFoundError('phishing_train.csv not found. Put the CSV files next to this notebook.')

train = pd.read_csv(data_dir + 'phishing_train.csv')
test  = pd.read_csv(data_dir + 'phishing_test.csv')
print(train.shape, test.shape)
x_train, y_train = train.drop(columns=['phishing']), train['phishing']
x_test,  y_test  = test.drop(columns=['phishing']),  test['phishing']
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)
pred = rf.predict(x_test)
print('accuracy', accuracy_score(y_test, pred), 'F1', f1_score(y_test, pred))
joblib.dump(rf, base + 'Models/rf_all.joblib')
joblib.dump(list(x_train.columns), base + 'Models/columns_all.joblib')
print('saved')

(45924, 112) (11481, 112)
accuracy 0.957059489591499 F1 0.9597846480137042
saved


In [ ]:
import sys; print(sys.executable)

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
lr=make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
cv_rf= cross_val_score(rf, x_train, y_train, cv=5, scoring='f1', n_jobs=-1)
cv_lr= cross_val_score(lr, x_train, y_train, cv=5, scoring='f1', n_jobs=-1)
print('RF CV F1:', cv_rf.mean().round(4), '+/-', cv_rf.std().round(4))
print('LR CV F1:', cv_lr.mean().round(4), '+/-', cv_lr.std().round(4))
lr.fit(x_train, y_train)
pred_lr= lr.predict(x_test)
print('LR ACCURACY', accuracy_score(y_test, pred_lr), 'F1', f1_score(y_test, pred_lr))
joblib.dump(lr, base + 'Models/lr_all.joblib')
print('saved')


RF CV F1: 0.9571 +/- 0.0016
LR CV F1: 0.9054 +/- 0.0028
LR ACCURACY 0.8989635049211742 F1 0.9059205190592052
saved


In [17]:
network_cols = ['time_response','domain_spf','asn_ip','time_domain_activation',
                'time_domain_expiration','qty_ip_resolved','qty_nameservers',
                'qty_mx_servers','ttl_hostname','tls_ssl_certificate',
                'qty_redirects','url_google_index','domain_google_index']
lexical_cols=[c for c in x_train.columns if c not in network_cols]
print(len(lexical_cols))

rf_lex=RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
lr_lex=make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
for name, model in[('RF LEXICAL', rf_lex), ('LR LEXICAL', lr_lex)]:
  cv = cross_val_score(model, x_train[lexical_cols], y_train, cv=5, scoring='f1', n_jobs=-1)
  model.fit(x_train[lexical_cols], y_train)
  p = model.predict(x_test[lexical_cols])
  print(name, 'CV F1:', cv.mean().round(4), '+/-', cv.std().round(4),
        '| test acc:', round(accuracy_score(y_test, p), 4),
          '| test F1:', round(f1_score(y_test, p), 4))

joblib.dump(rf_lex, base + 'Models/rf_lexical.joblib')
joblib.dump(lr_lex, base + 'Models/lr_lexical.joblib')
joblib.dump(lexical_cols, base + 'Models/columns_lexical.joblib')
print('saved')



98
RF LEXICAL CV F1: 0.9104 +/- 0.0032 | test acc: 0.9086 | test F1: 0.9129
LR LEXICAL CV F1: 0.8691 +/- 0.0035 | test acc: 0.8614 | test F1: 0.8715
saved


In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

all_cols = list(x_train.columns)

for name, cols in [('GB all', all_cols), ('GB lexical', lexical_cols)]:
    gb = HistGradientBoostingClassifier(random_state=42)
    cv = cross_val_score(gb, x_train[cols], y_train, cv=5, scoring='f1', n_jobs=-1)
    gb.fit(x_train[cols], y_train)
    p = gb.predict(x_test[cols])
    print(name, 'CV F1:', cv.mean().round(4), '+/-', cv.std().round(4),
          '| test acc:', round(accuracy_score(y_test, p), 4),
          '| test F1:', round(f1_score(y_test, p), 4))
    joblib.dump(gb, base + 'Models/gb_' + ('all' if name == 'GB all' else 'lexical') + '.joblib')
print('saved')

GB all CV F1: 0.9517 +/- 0.0012 | test acc: 0.9488 | test F1: 0.9518
GB lexical CV F1: 0.9033 +/- 0.0031 | test acc: 0.8991 | test F1: 0.904
saved
